# Sprint 2 · Grafo de conocimiento v1 + ontología mínima

**Tesis:** Diagnóstico de causalidad de fallas en una red telco usando grafos · Región **Pasco**, Perú.

Instancia la **ontología mínima** (`docs/03_ontologia.md`, `docs/ontologia_v1.ttl`) con la topología de `red.xlsx`
(hojas `nodos`, `topologia_ax`, `topologia_tx`) y ubica cada alarma en su equipo. La lógica está en `src/grafo.py` y las
vistas interactivas en `src/visual.py`.

**Vistas interactivas:** arrastra los nodos, acerca con la rueda, pasa el cursor para ver los datos, busca por código o
nombre, haz clic en la leyenda para ocultar un tipo y clic en un nodo para fijar su detalle. Cada vista también se guarda
como HTML en `reports/sprint2/` (se abre con doble clic, sin internet).

**Contenido**
1. Construcción del grafo · 2. Composición · 3. Validación · 4. Red a nivel de sitio · 5. Cluster a nivel de equipo ·
6. Jerarquía y criticidad · 7. Ruta aguas arriba · 8. Alarmas en el grafo · 9. Figuras para la tesis · 10. Exportación


## 1. Construcción del grafo

Reglas confirmadas con el NOC: el router de cada nodo distrital (AD) se engancha al router de transporte (TT) del mismo
sitio (salvo las excepciones de `data/raw/reglas_red.json`); el NOC Acceso se engancha al NOC Transporte; los dos OptiX OSN
forman un enlace óptico; la **RDNFO** es una ruta de respaldo. Los sectoriales quedan fuera.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.grafo import (construir_grafo, resumen_grafo, subgrafo_fisico, grafo_sitios, jerarquia_sitios,
                       ruta_al_nucleo, dependientes_por_sitio, red_propia, ubicar_alarmas, exportar_grafo)
from src.visual import vista_sitios, vista_cluster, mostrar, guardar_html

BASE_DIR = ROOT / "data" / "raw"                    # aquí están red.xlsx y las carpetas de ventanas
RED_XLSX = BASE_DIR / "red.xlsx"
REPORTES = ROOT / "reports" / "sprint2"             # vistas interactivas (.html)
FIGURAS = ROOT / "docs" / "figuras"                 # figuras estáticas (.png) para el documento
G = construir_grafo(str(RED_XLSX))

> **Resumen.** Lee las tres hojas de `red.xlsx` y construye el grafo. Imprime el total de nodos y aristas y los **avisos de calidad** (equipos sin código de nodo, equipos sin conexión física, etc.).

## 2. Composición del grafo (ontología instanciada)

In [ ]:
nodos, rels = resumen_grafo(G)
display(nodos)
display(rels)

> **Qué muestran las tablas.** *(Arriba)* cuántos nodos hay de cada **clase** de la ontología (Sitio, Equipo, Enlace) y subclase (router de acceso, IDU de microondas, enlace de fibra…). *(Abajo)* cuántas aristas hay de cada **relación**: `contiene` (sitio→equipo), `conexionLocal` (router↔IDU, OptiX↔router), `extremoDe` (equipo→enlace) y `engancha` (acceso→transporte y NOC Acceso→NOC Transporte).

## 3. Validación de la topología

Un grafo útil para el diagnóstico debe ser **conexo**: cada equipo tiene que poder alcanzar el núcleo de la red.

In [ ]:
F = subgrafo_fisico(G)
H = grafo_sitios(G)
J = jerarquia_sitios(G, H)
print(f"Subgrafo físico (equipos + enlaces): {F.number_of_nodes()} nodos | componentes: {nx.number_connected_components(F)}")
print(f"Grafo de sitios: {H.number_of_nodes()} sitios, {H.number_of_edges()} conexiones | componentes: {nx.number_connected_components(H)}")
clusters = {d.get('cluster') for _, d in H.nodes(data=True) if d.get('cluster')}
print("Acceso: cada cluster es un árbol →",
      all(nx.is_tree(H.subgraph([n for n, d in H.nodes(data=True) if d.get('cluster') == cl])) for cl in clusters))
print("\nAvisos:"); [print(" -", a) for a in G.graph["avisos"]]

> **Resumen.** Verifica que la red sea **una sola pieza conexa** (1 componente) a nivel de equipos y de sitios, y que cada cluster de acceso sea un **árbol** (sin ciclos, con el nodo distrital como raíz). Los avisos listan lo que falta completar en `red.xlsx`.

## 4. Red a nivel de sitio (interactiva)

In [ ]:
html = vista_sitios(G, H, J)
mostrar(html)
guardar_html(html, REPORTES / "grafo_red_sitios.html")

> **Qué muestra la vista.** Cada nodo es un **sitio** y cada línea una conexión. Azul: **transporte** (anillos de fibra); verde: **acceso** (◆ = nodo distrital, raíz de cada cluster); estrella roja: **NOC**; triángulo ámbar: **RDNFO**, con sus tramos de **respaldo** en línea punteada. El **tamaño** de cada sitio es proporcional a cuántos sitios dependen de él (criticidad). Pasa el cursor por un sitio para ver nombre, tipo, anillo/cluster, nivel en el árbol, saltos al NOC y criticidad.

## 5. Cluster de acceso a nivel de equipo (interactivo)

In [ ]:
CLUSTER = "CHACAYAN"                                # cambia por cualquier cluster de topologia_ax
html = vista_cluster(G, CLUSTER)
mostrar(html, alto=620)
guardar_html(html, REPORTES / f"grafo_cluster_{CLUSTER.replace(' ', '_')}.html")

> **Qué muestra la vista.** El cluster como lo describe el NOC, de arriba hacia abajo: **router de transporte** → **router del nodo distrital** → **IDU** (triángulos) → **enlace de microondas** (cuadrados) → IDU → router del nodo siguiente. Es el camino que recorrerá el motor para explicar una avalancha. Pasa el cursor por un equipo para ver su modelo, IP y sitio.

## 6. Jerarquía y criticidad (features topológicas)

Insumo de la tarjeta *Ingeniería de atributos*: **nivel en el árbol de acceso**, **saltos al NOC** por la ruta principal y
**dependientes** (sitios que se quedan sin camino al núcleo si ese sitio falla, contando el respaldo RDNFO).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
J["nivel_arbol_acceso"].dropna().astype(int).value_counts().sort_index().plot.bar(ax=ax, color="#2BA88A")
ax.set_title("Sitios de acceso por nivel en el árbol (0 = nodo distrital)")
ax.set_xlabel("nivel (saltos desde el nodo distrital)"); ax.set_ylabel("n sitios"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
print(f"Sitios críticos (su falla aísla a otros): {int(J['es_critico'].sum())} de {len(J)}")
display(J.sort_values("dependientes", ascending=False).head(8)[["sitio", "nombre", "red", "tipo_nodo", "saltos_al_nucleo", "dependientes"]])

> **Qué muestran la gráfica y la tabla.** La barra cuenta cuántos sitios de acceso hay en cada **nivel del árbol**. La tabla lista los sitios **más críticos**: los que, si fallan, dejan sin conexión a más sitios. Una alarma en uno de ellos es una buena candidata a explicar las avalanchas de los sitios que dependen de él.

In [ ]:
# Efecto de la RDNFO como respaldo: criticidad con y sin la ruta alternativa
d_con, d_sin = dependientes_por_sitio(H), dependientes_por_sitio(red_propia(H))
efecto = pd.DataFrame([{"sitio": s, "nombre": H.nodes[s].get("nombre"),
                        "dependientes_sin_RDNFO": d_sin.get(s, 0), "dependientes_con_RDNFO": d_con.get(s, 0)}
                       for s in set(d_sin) | set(d_con) if d_sin.get(s, 0) != d_con.get(s, 0)])
display(efecto.sort_values("dependientes_sin_RDNFO", ascending=False).head(8))

> **Qué muestra la tabla.** Los sitios cuya criticidad **baja gracias a la RDNFO**: sin el respaldo, su falla dejaría aislados a muchos sitios del bus de Oxapampa; con él, el bus mantiene una salida alternativa. Es la redundancia que el motor debe considerar: una falla con respaldo no necesariamente provoca una avalancha.

## 7. Ruta aguas arriba (interactiva)

Dado un sitio con alarmas, ¿por qué sitios pasa su conexión hacia el NOC? Las fallas en esos sitios son candidatas a causa
raíz. La **ruta principal** va por la red propia; con la RDNFO se obtiene la alternativa.

In [ ]:
SITIO = J.sort_values("saltos_al_nucleo", ascending=False)["sitio"].iloc[0]   # el más alejado del NOC (o escribe un código)
ruta = ruta_al_nucleo(G, SITIO, H)
print("Ruta principal     :", " → ".join(ruta or []))
print("Usando la RDNFO    :", " → ".join(ruta_al_nucleo(G, SITIO, H, evitar_externos=False) or []))
html = vista_sitios(G, H, J, resaltar_ruta=ruta, titulo=f"Ruta principal de {SITIO} al NOC")
mostrar(html)
guardar_html(html, REPORTES / f"grafo_ruta_{SITIO}.html")

> **Qué muestra la vista.** La misma red, con la **ruta principal** del sitio consultado pintada en rojo hasta el NOC. Es la base de la **regla topológica** del motor: si un sitio de esa ruta también tiene alarmas en la misma ventana, es mejor candidato a causa raíz que los sitios que dependen de él.

## 8. Alarmas en el grafo (`ocurreEn`)

Usa el consolidado del notebook 01 (sin sectoriales) y ubica cada alarma en su equipo del grafo.

In [ ]:
proc = ROOT / "data" / "processed" / "dataset_consolidado.parquet"
df = ubicar_alarmas(pd.read_parquet(proc), G) if proc.exists() else None
if df is None:
    print("No hay consolidado: ejecuta primero el notebook 01.")
else:
    print(f"Alarmas ubicadas en el grafo: {df['en_grafo'].mean():.1%} de {len(df):,}")
    por_sitio = df.groupby("CODIGO DE NODO").size()
    display(por_sitio.sort_values(ascending=False).head(10).rename("alarmas").to_frame())
    html = vista_sitios(G, H, J, alarmas_por_sitio=por_sitio.to_dict(), titulo="Dónde se concentran las alarmas")
    mostrar(html)
    guardar_html(html, REPORTES / "grafo_alarmas_por_sitio.html")

> **Qué muestran la tabla y la vista.** El porcentaje de alarmas ubicadas en un equipo del grafo (debe ser ~100 %), los **10 sitios con más alarmas** y la red coloreada por **volumen de alarmas** (gris = sin alarmas, amarillo → rojo = más alarmas). Permite ver si las alarmas se agrupan en una misma rama.

In [ ]:
# Cluster con más alarmas, a nivel de equipo: qué equipos levantaron alarmas
if df is not None:
    sitio_cluster = {d["codigo"]: d.get("cluster") for _, d in G.nodes(data=True) if d.get("clase") == "Sitio"}
    df["cluster"] = df["CODIGO DE NODO"].map(sitio_cluster)
    top = df["cluster"].replace("", pd.NA).dropna().value_counts()
    if len(top):
        cl = top.index[0]
        por_equipo = df[df["cluster"] == cl].groupby("NOMBRE DE EQUIPO EN RED").size().to_dict()
        html = vista_cluster(G, cl, alarmas_por_equipo=por_equipo, titulo=f"Cluster {cl}: equipos con alarmas")
        mostrar(html, alto=620)
        guardar_html(html, REPORTES / f"grafo_cluster_alarmas_{cl.replace(' ', '_')}.html")

> **Qué muestra la vista.** El cluster de acceso con más alarmas del período, a nivel de equipo: en **rojo** los equipos que levantaron alarmas (el tooltip indica cuántas). Sirve para ver cómo una falla se reparte entre routers e IDU de una misma rama.

## 9. Figuras estáticas para el documento de tesis

In [ ]:
FIGURAS.mkdir(parents=True, exist_ok=True)
col = {"tx": "#0B5D8F", "ax": "#2BA88A", "externo": "#F2A541"}
pos = nx.spring_layout(H, seed=42, k=0.18, iterations=200)
colores = ["#D1495B" if str(d.get("tipo_nodo", "")).startswith("NOC") else col.get(d.get("red"), "#999")
           for _, d in H.nodes(data=True)]
fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.35, width=0.8)
nx.draw_networkx_nodes(H, pos, ax=ax, node_color=colores, node_size=[25 + 18 * H.degree(n) for n in H], linewidths=0)
for lab, cc in [("Transporte", col["tx"]), ("Acceso", col["ax"]), ("RDNFO", col["externo"]), ("NOC", "#D1495B")]:
    ax.scatter([], [], c=cc, label=lab)
ax.legend(loc="upper left"); ax.set_title("Red de Pasco a nivel de sitio (grafo v1)"); ax.axis("off")
plt.tight_layout(); fig.savefig(FIGURAS / "grafo_red_sitios.png", dpi=200); plt.show()
print("Guardado:", FIGURAS / "grafo_red_sitios.png")

> **Resumen.** Versión **estática** (PNG, 200 dpi) de la red para insertar en el documento de tesis, que no admite gráficos interactivos. Se guarda en `docs/figuras/`.

## 10. Exportación

In [ ]:
ruta = exportar_grafo(G, str(ROOT / "data" / "processed"))
print("Grafo:", ruta, "y grafo_v1.json")
print("Vistas interactivas:", sorted(p.name for p in REPORTES.glob("grafo_*.html")))

> **Resumen.** Guarda el grafo en **GraphML** (Gephi / yEd) y **JSON** para el motor y el GraphRAG, y lista las vistas interactivas generadas en `reports/sprint2/`.

## Conclusiones del grafo v1

- La ontología mínima quedó **instanciada** con la topología real: sitios, equipos y enlaces unidos por `contiene`,
  `conexionLocal`, `extremoDe` y `engancha`, incluidos el enlace óptico entre los OptiX OSN y la RDNFO como respaldo.
- La red completa es **conexa** y cada cluster de acceso es un **árbol** con raíz en su nodo distrital: permite razonar
  «aguas arriba / aguas abajo», base del diagnóstico de causa raíz.
- La **criticidad** (sitios dependientes) cuantifica el impacto de cada sitio y muestra el efecto de la redundancia
  (RDNFO): son features topológicas listas para el motor.
- Cada alarma se ubica en su equipo del grafo (`ocurreEn`), con el puerto como atributo de la alarma (OE3).
